Data Loading Notebook
===

Prepare data for the analysis. The raw data is the public FAIR Universe HiggsML dataset (one 220M-row parquet file). This notebook streams it row group by row group, keeps a configurable fraction of each process, applies the configured systematic variations chunk by chunk and writes one `.root` ntuple per variation with one tree per process. It never loads the full dataset, so peak memory stays at a few copies of one chunk (set `data.chunk_rows` in `config.pipeline.yaml` to tune it).

It calls the very same functions as `scripts/data_loader.py`, which is what the Snakemake workflow runs; the notebook only breaks the script's `main()` into steps so each one can be inspected.

This notebook is only to be run once to get the FAIR Universe dataset into `.root` ntuples. The rest of the workflow is independent of this.

#### Note: you can choose to skip this step (and notebook 2) entirely by downloading a pre-processed `saved_datasets.tar.gz` from [https://cernbox.cern.ch/s/mltUDvzKdisEEpJ](https://cernbox.cern.ch/s/mltUDvzKdisEEpJ) and running `tar -xvf saved_datasets.tar.gz`. The notebook detects existing ntuples in the output directory and then skips the download and processing automatically, so re-running it is harmless.


In [1]:
import os, sys
import numpy as np
import pandas as pd
import uproot
import matplotlib.pyplot as plt
import mplhep as hep

from nsbi_common_utils.configuration import load_pipeline_config

# The streaming loader lives in scripts/data_loader.py (the CLI the Snakemake workflow runs); this notebook calls the same functions step by step.
sys.path.append("scripts")
from data_loader import open_source, resolve_fractions, select_events, iter_chunks, RootWriter
from HiggsML.systematics import systematics

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

hep.style.use(hep.style.ATLAS)


Configuration
--

Everything below is driven by the `data_loader` block of `config.pipeline.yaml`: which processes to keep and what fraction of each, the event selections, the systematic variations to generate, and the output directory (`${NSBI_DATA_DIR:-./saved_datasets}`).


In [2]:
data_config = load_pipeline_config("config.pipeline.yaml")["data_loader"]
data_cfg    = data_config["data"]
pre_cfg     = data_config["preprocess"]

processes  = list(data_cfg["processes"])
seed       = int(pre_cfg["seed"])
balance    = bool(pre_cfg.get("balance", False))
selections = pre_cfg["selections"]
chunk_rows = int(data_cfg.get("chunk_rows", 2_000_000))
output_dir = data_config["output"]["dir"]

print(f"Processes to keep: {processes}")
print(f"Event selections: {selections}")
print(f"Output directory: {output_dir}")


Processes to keep: ['htautau', 'ztautau', 'ttbar']
Event selections: DER_mass_transverse_met_lep <= 250.0 and DER_mass_vis <= 500.0 and DER_sum_pt <= 1000 and DER_pt_tot <= 250 and DER_deltar_had_lep <= 4.5 and DER_pt_h <= 400 and DER_pt_ratio_lep_had <= 9.0

Output directory: ./saved_datasets/


In [3]:
# One entry per output file: the nominal sample plus every configured systematic variation. The shared seed is injected into each variation (TES/JES are deterministic, the seed only matters for the soft-MET smearing).
syst_settings = {name: dict(syst_args) for name, syst_args in data_config["systematics"].items()}
for syst_args in syst_settings.values():
    syst_args["seed"] = seed

variations = {"nominal": {}}
variations.update(syst_settings)
print(f"Variations to write: {list(variations)}")


Variations to write: ['nominal', 'TES_up', 'TES_dn', 'JES_up', 'JES_dn']


Skip switch
--

If every output ntuple already exists (from the pre-processed archive, an earlier run of this notebook, or the Snakemake workflow) nothing is downloaded or re-streamed: the heavy cells below become no-ops and only the summary at the end runs.


In [4]:
output_files  = {name: os.path.join(output_dir, f"dataset_{name}.root") for name in variations}
datasets_exist = all(os.path.exists(path) for path in output_files.values())

if datasets_exist:
    print("Saved datasets already exist - download and processing are skipped:")
else:
    print("Saved datasets not found - the raw dataset will be streamed and processed into:")
for path in output_files.values():
    print(f"  {'[found]  ' if os.path.exists(path) else '[missing]'} {path}")


Saved datasets already exist - download and processing are skipped:
  [found]   ./saved_datasets/dataset_nominal.root
  [found]   ./saved_datasets/dataset_TES_up.root
  [found]   ./saved_datasets/dataset_TES_dn.root
  [found]   ./saved_datasets/dataset_JES_up.root
  [found]   ./saved_datasets/dataset_JES_dn.root


Open the source dataset
--

`download_dataset` fetches and extracts the public archive unless `./data.zip` or an extracted `./data` directory already exists in the working directory, in which case it is reused. The parquet file is only opened, not read.

The challenge reserves the first `test_size` fraction of rows as its held-out test set (the same split `HiggsML.datasets.Data` uses). Those rows are never used for training here.


In [5]:
if not datasets_exist:
    pf, metadata = open_source(data_cfg["url"])
    total_rows   = int(metadata.get("total_rows", pf.metadata.num_rows))

    test_size = float(data_cfg.get("test_size", 0.3))
    if not 0.0 <= test_size < 1.0:
        raise ValueError(f"data.test_size must be in [0, 1), got {test_size}")
    n_test = int(test_size * total_rows)

    print(f"Source: {total_rows} rows in {pf.num_row_groups} row groups; {chunk_rows} rows per chunk")
    print(f"Reserved test region: first {n_test} rows (test_size={test_size}); training region: {total_rows - n_test} rows")


Select events
--

A weights-only pass over the file draws one seeded uniform number per row and keeps an event of process `p` iff `u < fractions[p]`. The pass also accumulates, per process, the event counts and weight sums before and after the selection, which the next step uses to restore the physical yields. With `preprocess.balance: true` the fractions are reduced so every process keeps the same expected number of events.


In [6]:
if not datasets_exist:
    fractions = resolve_fractions(data_cfg, processes)
    print(f"Per-process keep fractions: {fractions}")
    stats, w_region, masks = select_events(pf, fractions, seed, n_test)

    if balance:
        # Reduce every fraction so all processes keep the same expected count as the smallest one. Re-running the pass with the same seed keeps the selection nested inside the first one.
        target = min(stats[p]["n_kept"] for p in processes)
        fractions = {p: fractions[p] * target / stats[p]["n_kept"] for p in processes}
        print(f"Balancing enabled: every process reduced to ~{target} events; effective fractions {fractions}")
        stats, w_region, masks = select_events(pf, fractions, seed, n_test)


Weight factors
--

Two factors are applied to the kept events. A global one reproduces the `HiggsML` convention that the training set's total weight equals the metadata `sum_weights` regardless of `test_size`. A per-process one, (weight of every event of `p` in the training region) / (weight of the kept events of `p`), undoes the subsampling so each process keeps its physical yield.


In [7]:
if not datasets_exist:
    sum_weights   = float(metadata.get("sum_weights", 0.0))
    global_factor = sum_weights / w_region if sum_weights > 0.0 and w_region > 0.0 else 1.0
    print(f"Training region holds {w_region:.1f} of the metadata sum_weights {sum_weights:.1f}: global weight factor {global_factor:.6f}\n")

    factor = {}
    print("Per-process selection (training region only):")
    for p in processes:
        s = stats[p]
        if s["w_kept"] <= 0.0:
            raise ValueError(f"No events of process '{p}' selected; check data.fractions")
        factor[p] = global_factor * s["w_all"] / s["w_kept"]
        print(f"  {p:10s} all={s['n_all']:>11d}  kept={s['n_kept']:>10d} ({100.0 * s['n_kept'] / max(s['n_all'], 1):6.2f}%)  "
              f"region yield={s['w_all']:12.1f} -> {global_factor * s['w_all']:12.1f}  weight factor={factor[p]:.6f}  "
              f"[test region: {s['n_test']} events, weight {s['w_test']:.1f}]")


Stream, vary, select and write
--

Kept rows are read in runs of whole row groups of about `chunk_rows` source rows. Each chunk is reweighted, passed through `HiggsML.systematics.systematics()` once per variation, cut with the configured selections and appended to the per-process trees of the corresponding output file. The files are written under a temporary name and moved into place when complete.


In [8]:
if not datasets_exist:
    os.makedirs(output_dir, exist_ok=True)
    writers = {name: RootWriter(path) for name, path in output_files.items()}

    n_seen = 0
    try:
        for ichunk, (n_read, df_chunk) in enumerate(iter_chunks(pf, chunk_rows, masks)):
            n_seen += n_read
            if df_chunk.empty:
                print(f"Chunk {ichunk}: no selected events ({n_seen}/{total_rows} rows read)")
                continue
            df_chunk["weights"] = df_chunk["weights"].to_numpy() * df_chunk["detailed_labels"].map(factor).to_numpy()
            print(f"Chunk {ichunk}: {len(df_chunk)} selected events ({n_seen}/{total_rows} rows read)")

            for name, syst_args in variations.items():
                df_var = systematics(data_set=df_chunk, dopostprocess=False, **syst_args)
                df_var = df_var.query(selections)
                for p in processes:
                    df_p = df_var[df_var["detailed_labels"] == p]
                    if len(df_p) == 0:
                        continue
                    columns_to_keep = [c for c in df_p.columns if c != "detailed_labels"]
                    writers[name].write(f"tree_{p}", df_p[columns_to_keep])
                del df_var
            del df_chunk
    finally:
        for writer in writers.values():
            writer.close()

    if n_seen != total_rows:
        raise RuntimeError(f"Read {n_seen} rows but the source has {total_rows}")

    for name, writer in writers.items():
        for tree_name, (n, s) in sorted(writer.stats.items()):
            print(f"  {name:10s} {tree_name:14s} events={n:>10d}  sum of weights={s:12.1f}")
        print(f"Wrote {writer.path}")


In [9]:
# Remove the extracted raw dataset so it is not left next to the notebook (on a cluster it would otherwise be shipped back with the job output).
import shutil
if not datasets_exist:
    for leftover in ("data", "data.zip"):
        if os.path.islink(leftover) or os.path.isfile(leftover):
            os.remove(leftover)
        elif os.path.isdir(leftover):
            shutil.rmtree(leftover, ignore_errors=True)


Summary of the nominal ntuple
--

Events and physical yields per process in the nominal sample, read back from the file that the rest of the workflow consumes.


In [10]:
with uproot.open(output_files["nominal"]) as f:
    rows = []
    for tree_name in sorted(f.keys(cycle=False)):
        tree = f[tree_name]
        w = tree["weights"].array(library="np")
        rows.append({"tree": tree_name, "events": tree.num_entries, "sum of weights": w.sum(),
                     "min weight": w.min(), "max weight": w.max()})
summary = pd.DataFrame(rows).set_index("tree")
summary


,events,sum of weights,min weight,max weight
tree,,,,
tree_htautau,10143267,1.009691e+03,0.000079,0.000102
tree_ttbar,4218739,4.299578e+04,0.008996,0.015810
tree_ztautau,29501888,1.002118e+06,0.033969,0.033969
